# 04 - Ciclos y colecciones en R

## Proposito de la sesion
Este notebook es una guia para aprender desde cero como recorrer datos y organizar colecciones en R.
La meta no es memorizar sintaxis, sino entender cuando usar cada herramienta y como evitar errores frecuentes.

## Objetivos de aprendizaje
Al finalizar la sesion, deberias poder:

1. Explicar la diferencia entre `for` y `while` en R.
2. Usar vectores y listas como colecciones base en R.
3. Identificar el analogo en R de una "tupla" de Python.
4. Resolver problemas con ciclos sin caer en copiado mecanico.
5. Justificar cuando conviene vectorizar en lugar de iterar manualmente.

## Antes de empezar: idea central en R
R esta diseniado para trabajar con vectores.

- Los ciclos (`for`, `while`) existen y son utiles para aprender logica y para ciertos algoritmos.
- Pero muchas tareas en R se resuelven mejor con operaciones vectorizadas (`sum`, `mean`, `ifelse`, `sapply`, etc.).

En esta sesion veremos ambos enfoques para que compares criterio, claridad y riesgos.

In [ ]:
# Verificacion minima del entorno R
R.version.string
sessionInfo()

## 1) Colecciones en R desde cero
En esta clase usaremos dos colecciones principales:

- **Vector atomico**: todos los elementos del mismo tipo (numeric, character, logical).
- **Lista (`list`)**: puede mezclar tipos y estructuras.

Nota importante:
- En Python se habla de listas y tuplas.
- En R no existe una "tupla" nativa equivalente. El analogo practico suele ser un vector pequeno nombrado o una lista pequena tratada como registro fijo.

In [ ]:
# Ejemplo de vector atomico: todos los elementos son numericos
calificaciones <- c(91, 78, 85, 100, 66)

# Ejemplo de vector de texto
nombres <- c("Ana", "Luis", "Marta", "Jorge", "Sara")

# Ejemplo de lista: mezcla tipos (texto, numero, vector)
ficha_grupo <- list(
  materia = "Programacion en R",
  semestre = 2,
  aprobado = c(TRUE, TRUE, TRUE, TRUE, FALSE)
)

# str() ayuda a inspeccionar estructura interna de forma clara
str(calificaciones)
str(ficha_grupo)

## 2) Ciclo `for`: recorrer una coleccion
`for` recorre elementos de una secuencia conocida.

Patron mental:
1. Definir que se va a recorrer.
2. Definir que resultado quieres construir.
3. Preasignar memoria cuando el resultado tendra tamano fijo.

In [ ]:
# Problema: calcular cuadrados de 1 a n
n <- 8

# Preasignamos un vector numerico para evitar crecimiento dinamico en cada iteracion
cuadrados <- numeric(n)

for (i in seq_len(n)) {
  # En cada paso guardamos i^2 en la posicion i
  cuadrados[i] <- i^2
}

cuadrados

In [ ]:
# Ejemplo de for sobre una lista heterogenea
objeto <- list(
  nombre = "sensor_A",
  activo = TRUE,
  mediciones = c(12.1, 11.8, 12.4)
)

for (clave in names(objeto)) {
  # [[ ]] extrae el elemento de la lista; [ ] regresa sublista
  valor <- objeto[[clave]]
  cat("Campo:", clave, "| Clase:", class(valor), "\n")
}

## 3) Ciclo `while`, `break` y `next`
`while` repite mientras una condicion sea `TRUE`.

- `break`: detiene el ciclo de inmediato.
- `next`: salta al siguiente paso (analogo de `continue` en Python).

`while` es util cuando no conoces de antemano cuantas iteraciones haras.

In [ ]:
# Simulamos lanzamientos hasta alcanzar una suma objetivo
set.seed(2026)

objetivo <- 20
suma <- 0
intentos <- 0
limite_intentos <- 30

while (suma < objetivo) {
  intentos <- intentos + 1

  # Corte de seguridad: se evalua antes de cualquier next
  if (intentos > limite_intentos) {
    break
  }

  tiro <- sample(1:6, size = 1)

  # Si sale 1, ignoramos el tiro para este ejemplo didactico
  if (tiro == 1) {
    next
  }

  suma <- suma + tiro
}

list(suma_final = suma, intentos = intentos)

## 4) Tema adaptado: "tuplas" de Python en R
Como R no tiene una tupla nativa inmutable, puedes usar dos analogos segun contexto:

1. **Vector nombrado** para datos pequenos del mismo tipo.
2. **Lista pequena** para mezclar tipos en un registro corto.

Practica recomendada en clase: tratar estos objetos como "no mutables por convencion" cuando representen coordenadas, parametros o configuraciones.

In [ ]:
# Analogo 1: vector nombrado (todo numerico)
punto <- c(x = 3.5, y = -1.2)
punto

# Analogo 2: lista corta (tipos mezclados)
persona <- list(nombre = "Elena", edad = 21)
persona

# R permite modificar ambos objetos, pero en muchos modelos conviene
# no cambiarlos una vez definidos para mantener consistencia conceptual.


## 5) Errores comunes y puntos que se deben cuidar
### Errores tecnicos frecuentes
- Usar `1:n` cuando `n` puede ser 0. Mejor `seq_len(n)`.
- Olvidar actualizar la condicion en `while` y crear un ciclo infinito.
- Crecer vectores dentro del `for` sin preasignar memoria.
- Mezclar tipos sin querer (coercion automatica en vectores atomicos).
- Confundir `NA` con `FALSE` en condiciones.

### Puntos didacticos a cuidar al enseniar
- Explicar primero estructura de datos y luego ciclos (no al reves).
- Comparar siempre una solucion con ciclo contra una vectorizada.
- Pedir a estudiantes justificar decisiones, no solo mostrar salida correcta.
- Incluir casos borde: vector vacio, valores faltantes, umbrales extremos.
- Evaluar legibilidad y validaciones, no solo que "corra".

In [ ]:
# Micro-ejemplos de cuidado

# 1) Caso n = 0: seq_len evita indices invalidos
n <- 0
seq_len(n)  # devuelve integer(0), seguro para for

# 2) Cuidado con NA en condicionales
valor <- NA

if (is.na(valor)) {
  mensaje <- "Hay un dato faltante (NA), no se puede decidir aun."
} else if (valor > 10) {
  mensaje <- "Mayor que 10"
} else {
  mensaje <- "Menor o igual a 10"
}

mensaje

## 6) Ejemplo integrador: resumen de notas por estudiante
Problema:
Tienes una lista donde cada estudiante guarda varias notas.
Quieres calcular promedio por estudiante y detectar si aprueba (>= 70).

Primero lo hacemos con `for` para practicar razonamiento paso a paso.

In [ ]:
# Datos de ejemplo
notas <- list(
  Ana = c(80, 75, 92),
  Luis = c(60, 70, 65),
  Marta = c(100, 95, 98),
  Jorge = c(NA, 72, 68)
)

promedios <- numeric(length(notas))
nombres_est <- names(notas)

for (i in seq_along(notas)) {
  # na.rm = TRUE evita que un NA invalide todo el promedio
  promedios[i] <- mean(notas[[i]], na.rm = TRUE)
}

resultado <- data.frame(
  estudiante = nombres_est,
  promedio = round(promedios, 2),
  aprueba = promedios >= 70
)

resultado

In [ ]:
# Version vectorizada del mismo problema (para comparar estilo)
promedios_vec <- sapply(notas, mean, na.rm = TRUE)

resultado_vec <- data.frame(
  estudiante = names(promedios_vec),
  promedio = round(as.numeric(promedios_vec), 2),
  aprueba = as.numeric(promedios_vec) >= 70
)

resultado_vec

## 7) Ejercicios para pensar (no para copiar)
### Instrucciones
Para cada ejercicio:
1. Escribe primero tu estrategia en texto (3-5 lineas).
2. Identifica un caso borde y como lo controlarias.
3. Implementa en R y compara con una alternativa vectorizada cuando sea posible.

### Ejercicio 1: Conteo con criterio compuesto
Dado un vector de temperaturas, cuenta cuantas estan entre 18 y 25 inclusive,
pero ignora valores `NA` y valores negativos (errores de sensor).

Pregunta de razonamiento: 
- Que cambia si usas `for` vs filtrado vectorizado?

### Ejercicio 2: Corte temprano con `break`
Simula lanzamientos de un dado hasta que la suma acumulada supere 50.
Reporta numero de tiros y promedio por tiro.

Pregunta de razonamiento:
- Que condicion de seguridad agregarias para evitar un bucle infinito?

### Ejercicio 3: Analogo de tupla para coordenadas
Representa 5 puntos 2D en R con un analogo apropiado de tupla.
Calcula la distancia al origen de cada punto y decide cual esta mas lejos.

Pregunta de razonamiento:
- Que estructura elegirias y por que: vector nombrado, lista o data.frame?

### Ejercicio 4: Validacion de entrada
Escribe una funcion que reciba un vector numerico y regrese su media recortada
(quitando el valor maximo y minimo), con validaciones:
- Error si hay menos de 3 datos validos.
- Manejo explicito de `NA`.

Pregunta de razonamiento:
- Que tipo de error deberia lanzar y por que?

In [ ]:
# Espacio de trabajo para resolver ejercicios
# Sugerencia: resolver uno por uno y documentar decisiones.


## 8) Cierre de la sesion
Checklist rapido de dominio:

- Puedes explicar con tus palabras cuando usar `for` y cuando `while`.
- Sabes elegir entre vector y lista segun tipo de datos.
- Entiendes que en R no hay tupla nativa y sabes su analogo practico.
- Tomas en cuenta casos borde (`NA`, vector vacio, limites de ciclo).
- Puedes comparar una solucion iterativa contra una vectorizada con criterio.